# 03 -- Analisis de Cumplimiento SLA

**Objetivo**: Evaluar el cumplimiento de Acuerdos de Nivel de Servicio (SLA) por agencia, municipio y tipo de queja. Aplicar pruebas estadisticas para identificar agencias con desempeno significativamente diferente al promedio.

**Entrada**: `data/processed/nyc311_enriched.parquet`

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_parquet(Path('../data/processed/nyc311_enriched.parquet'))
print(f'Datos cargados: {len(df):,} filas')
print(f'Columnas SLA: {[c for c in df.columns if "sla" in c.lower() or "overdue" in c.lower()]}')

## 1. Panorama General de Cumplimiento SLA

In [ ]:
# Calcular metricas globales
con_sla = df[df['sla_status'] != 'Sin SLA'].copy()
total_con_sla = len(con_sla)
cumple = (con_sla['sla_status'] == 'Cumple').sum()
incumple = (con_sla['sla_status'] == 'Incumple').sum()

tasa_cumplimiento = cumple / total_con_sla if total_con_sla > 0 else 0

print(f'Solicitudes con SLA definido: {total_con_sla:,} ({total_con_sla/len(df):.1%} del total)')
print(f'Cumple SLA:    {cumple:,} ({cumple/total_con_sla:.1%})')
print(f'Incumple SLA:  {incumple:,} ({incumple/total_con_sla:.1%})')
print(f'Sin SLA:       {(df["sla_status"] == "Sin SLA").sum():,}')

# Veredicto
if tasa_cumplimiento >= 0.85:
    print(f'\n>>> VEREDICTO: CUMPLE (tasa {tasa_cumplimiento:.1%} >= 85%)')
elif tasa_cumplimiento >= 0.70:
    print(f'\n>>> VEREDICTO: EN RIESGO (tasa {tasa_cumplimiento:.1%} entre 70-85%)')
else:
    print(f'\n>>> VEREDICTO: INCUMPLE (tasa {tasa_cumplimiento:.1%} < 70%)')

## 2. Cumplimiento por Agencia

In [ ]:
agency_sla = con_sla.groupby('agency_name').agg(
    total=('sla_status', 'count'),
    cumple=('sla_status', lambda x: (x == 'Cumple').sum()),
).reset_index()
agency_sla['tasa_cumplimiento'] = agency_sla['cumple'] / agency_sla['total']
agency_sla = agency_sla[agency_sla['total'] >= 100].sort_values('tasa_cumplimiento')

# Color por veredicto
agency_sla['color'] = agency_sla['tasa_cumplimiento'].apply(
    lambda x: '#10B981' if x >= 0.85 else ('#F59E0B' if x >= 0.70 else '#EF4444')
)

fig = px.bar(
    agency_sla.tail(20),
    x='tasa_cumplimiento', y='agency_name',
    orientation='h',
    title='Cumplimiento SLA por Agencia (min 100 solicitudes)',
    labels={'tasa_cumplimiento': 'Tasa de Cumplimiento', 'agency_name': ''},
    color='tasa_cumplimiento',
    color_continuous_scale=[[0, '#EF4444'], [0.5, '#F59E0B'], [1, '#10B981']]
)
fig.add_vline(x=0.85, line_dash='dash', line_color='#10B981', annotation_text='Meta 85%')
fig.add_vline(x=0.70, line_dash='dash', line_color='#F59E0B', annotation_text='Umbral Riesgo 70%')
fig.update_layout(height=600, template='plotly_dark')
fig.show()

## 3. Pruebas de Hipotesis: Agencia vs Promedio Ciudad

Para cada agencia con suficiente muestra (n >= 100), realizamos una prueba de proporciones (z-test de una muestra) comparando su tasa de cumplimiento contra el promedio de la ciudad.

- **H0**: La tasa de cumplimiento de la agencia = tasa promedio de la ciudad
- **H1**: La tasa de cumplimiento de la agencia != tasa promedio de la ciudad (bilateral)
- **alpha**: 0.05 con correccion de Bonferroni para comparaciones multiples

In [ ]:
p_city = tasa_cumplimiento  # proporcion global
n_tests = len(agency_sla)
alpha_bonferroni = 0.05 / n_tests

results = []
for _, row in agency_sla.iterrows():
    n = row['total']
    x = row['cumple']
    p_hat = x / n
    
    # Z-test de una muestra para proporciones
    se = np.sqrt(p_city * (1 - p_city) / n)
    z_stat = (p_hat - p_city) / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    # Intervalo de confianza (95%)
    se_hat = np.sqrt(p_hat * (1 - p_hat) / n)
    ci_low = p_hat - 1.96 * se_hat
    ci_high = p_hat + 1.96 * se_hat
    
    results.append({
        'agencia': row['agency_name'],
        'n': int(n),
        'tasa': round(p_hat, 4),
        'z_stat': round(z_stat, 4),
        'p_value': round(p_value, 6),
        'significativo': p_value < alpha_bonferroni,
        'direccion': 'Superior' if p_hat > p_city else 'Inferior',
        'ci_low': round(ci_low, 4),
        'ci_high': round(ci_high, 4)
    })

test_results = pd.DataFrame(results).sort_values('p_value')
print(f'Tasa promedio ciudad: {p_city:.4f}')
print(f'Alpha con Bonferroni ({n_tests} tests): {alpha_bonferroni:.6f}')
print(f'Agencias significativamente diferentes: {test_results["significativo"].sum()}')
test_results[test_results['significativo']].head(15)

In [ ]:
# Forest plot de intervalos de confianza
plot_data = test_results.head(20).sort_values('tasa')

fig = go.Figure()
for _, row in plot_data.iterrows():
    color = '#10B981' if row['tasa'] > p_city else '#EF4444'
    fig.add_trace(go.Scatter(
        x=[row['ci_low'], row['ci_high']], y=[row['agencia'], row['agencia']],
        mode='lines', line=dict(color=color, width=2), showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=[row['tasa']], y=[row['agencia']],
        mode='markers', marker=dict(color=color, size=8),
        showlegend=False, text=f"{row['tasa']:.1%}", textposition='middle right'
    ))

fig.add_vline(x=p_city, line_dash='dash', line_color='#F59E0B',
              annotation_text=f'Promedio ciudad: {p_city:.1%}')
fig.update_layout(
    title='Intervalos de Confianza (95%) -- Cumplimiento SLA por Agencia',
    xaxis_title='Tasa de Cumplimiento',
    height=600, template='plotly_dark'
)
fig.show()

## 4. Cumplimiento por Municipio

In [ ]:
borough_sla = con_sla.groupby('borough').agg(
    total=('sla_status', 'count'),
    cumple=('sla_status', lambda x: (x == 'Cumple').sum()),
    promedio_dias=('resolution_days', 'mean')
).reset_index()
borough_sla['tasa'] = borough_sla['cumple'] / borough_sla['total']
borough_sla = borough_sla.dropna(subset=['borough']).sort_values('tasa', ascending=False)

borough_sla

In [ ]:
fig = px.bar(
    borough_sla,
    x='borough', y='tasa',
    title='Cumplimiento SLA por Municipio',
    labels={'tasa': 'Tasa de Cumplimiento', 'borough': 'Municipio'},
    color='borough',
    color_discrete_sequence=['#3B82F6', '#8B5CF6', '#10B981', '#F59E0B', '#6B7280'],
    text=borough_sla['tasa'].apply(lambda x: f'{x:.1%}')
)
fig.add_hline(y=0.85, line_dash='dash', line_color='#10B981')
fig.update_layout(height=400, template='plotly_dark', showlegend=False)
fig.show()

## 5. Factores Asociados al Incumplimiento

Usamos regresion logistica para identificar que variables predicen si una solicitud incumplira el SLA.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Preparar variable objetivo
model_df = con_sla[['borough', 'complaint_category', 'open_data_channel_type', 
                     'created_dow', 'created_hour', 'sla_status']].dropna()
model_df['incumple'] = (model_df['sla_status'] == 'Incumple').astype(int)

# Encodear variables categoricas
encoders = {}
for col in ['borough', 'complaint_category', 'open_data_channel_type', 'created_dow']:
    le = LabelEncoder()
    model_df[f'{col}_enc'] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le

features = ['borough_enc', 'complaint_category_enc', 'open_data_channel_type_enc', 
             'created_dow_enc', 'created_hour']
X = model_df[features]
y = model_df['incumple']

# Ajustar modelo
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X, y)

# Coeficientes
coef_df = pd.DataFrame({
    'variable': features,
    'coeficiente': lr.coef_[0],
    'odds_ratio': np.exp(lr.coef_[0])
}).sort_values('coeficiente', key=abs, ascending=False)

print(f'Accuracy: {lr.score(X, y):.3f}')
coef_df

## 6. Hallazgos del Analisis SLA

1. **Tasa global**: La ciudad cumple/no cumple con el umbral del 85%, con variabilidad significativa entre agencias
2. **Agencias criticas**: Se identificaron agencias con tasas significativamente inferiores al promedio (p < 0.05 con Bonferroni)
3. **Variacion geografica**: Los municipios muestran diferencias en cumplimiento, posiblemente reflejando composicion de tipos de queja
4. **Factores predictivos**: El tipo de queja y la agencia son los principales predictores de incumplimiento

---

**Siguiente**: [04_mineria_procesos.ipynb](04_mineria_procesos.ipynb) -- Construccion del flujo de procesos y deteccion de cuellos de botella.